# K-pop Narrative & Visual Mapping Pipeline

**Two-branch pipeline** — Genius에 가사/페이지가 있으면 "아는 곡" 분기,
없으면 "신곡" 분기로 라우팅한다.

## 데이터 소스
| 역할 | 소스 |
|---|---|
| 정량 feature | Librosa (로컬 음원 분석) |
| 분기 판정 + 가사/메타 | Genius API |
| 음원 1순위 | iTunes Search API (30초 m4a 프리뷰, 무료/무인증) |
| 음원 2순위 | YouTube URL (수동 입력) → yt-dlp |
| 분류 LLM | Google Gemini (text + audio) |

## 분류 출력 (3 카테고리)
1. **Sonic Texture** — 8개 중 1개
2. **Narrative Archetype** — 7개 중 1개
3. **Visual Symbol (3개) + Color Palette** — main + sub 2

## 두 분기
- **Known song** (Genius hit) — librosa 정량 + Genius narrative → Gemini text로 분류
- **New song** (Genius miss) — librosa 정량 + 오디오 파일 자체 → Gemini audio로 분류

## 사전 준비
리포 루트의 `.env`:
```
GENIUS_ACCESS_TOKEN=...
GEMINI_API_KEY=...
```

필요 패키지: `lyricsgenius`, `google-genai`, `python-dotenv`, `librosa`, `requests`, `yt-dlp`

In [1]:
%pip install -q lyricsgenius google-genai python-dotenv librosa requests yt-dlp


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Imports & 환경 변수

In [2]:
import json
import os
import re
import tempfile
from pathlib import Path
from typing import Literal, Optional

from dotenv import load_dotenv

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "music_analysis" else Path.cwd()
load_dotenv(REPO_ROOT / ".env")

GENIUS_ACCESS_TOKEN = os.getenv("GENIUS_ACCESS_TOKEN")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

DB_PATH = REPO_ROOT / "music_analysis" / "kpop_visual_db.json"
TMP_AUDIO_DIR = REPO_ROOT / "samples" / "_tmp"
TMP_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("keys loaded:", {
    "genius": bool(GENIUS_ACCESS_TOKEN),
    "gemini": bool(GEMINI_API_KEY),
})

REPO_ROOT: /Users/yuli/Documents/GitHub/0_Exp/Spatial_Audio_Visualization
keys loaded: {'genius': True, 'gemini': True}


## 2. 분류 표준 + 시스템 프롬프트

In [3]:
SONIC_TEXTURES = [
    "Synth-pop", "Brass-Heavy", "Glitch", "Orchestral",
    "Acoustic", "Rock-chic", "Hip-hop Beat", "Dreamy Pad",
]

NARRATIVE_ARCHETYPES = [
    "High-teen", "Cyberpunk", "Ethereal", "Narcissistic",
    "Mala-taste (Spicy)", "Retro-nostalgia", "Gothic-horror",
]

SYSTEM_PROMPT = f"""Role: K-pop Narrative & Visual Mapping Specialist

Task: 제공된 정량 데이터(Librosa)와 정성 데이터(Genius 가사/메타 또는 오디오)를 교차 분석하여,
K-pop 오디오 비주얼라이제이션용 표준 JSON을 생성하라.

Classification Standard (Consistency):
- Sonic Texture: {SONIC_TEXTURES} 중 1개 선택.
- Narrative Archetype: {NARRATIVE_ARCHETYPES} 중 1개 선택.
- Visual Symbol: 텍스트/오디오에서 도출된 시각적 오브젝트 3가지 (동물, 사물, 자연물 등).
- Color Mood: 메인 컬러(Hex 1개), 서브 컬러(Hex 2개).
  Librosa의 valence/energy를 색상의 명도/채도에 반영할 것.

Output Format (Strict JSON, no markdown, no commentary):
{{
  "sonic_texture": "",
  "narrative_archetype": "",
  "visual_symbols": ["", "", ""],
  "color_palette": {{"main": "#", "sub": ["#", "#"]}},
  "reasoning_brief": "분류 이유 요약 (한글, 2-3문장)"
}}

Constraint: 오직 JSON만 출력하고 다른 부연 설명은 하지 않는다."""

## 3. 로컬 캐시 DB

In [4]:
def _normalize_key(title: str, artist: str) -> str:
    return f"{title.strip().lower()}__{artist.strip().lower()}"

def _load_db() -> dict:
    if not DB_PATH.exists():
        return {}
    try:
        return json.loads(DB_PATH.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        return {}

def _save_db(db: dict) -> None:
    DB_PATH.parent.mkdir(parents=True, exist_ok=True)
    DB_PATH.write_text(json.dumps(db, ensure_ascii=False, indent=2), encoding="utf-8")

def get_from_cache(title: str, artist: str) -> Optional[dict]:
    return _load_db().get(_normalize_key(title, artist))

def save_to_cache(entry: dict) -> None:
    db = _load_db()
    db[_normalize_key(entry["metadata"]["title"], entry["metadata"]["artist"])] = entry
    _save_db(db)

## 4. 음원 확보 — iTunes 1순위, YouTube 2순위

### 4-a. iTunes Search API (무료/무인증)
30초 m4a 프리뷰만 받지만 신곡 sonic texture 분류엔 충분하다.
Apple Music 카탈로그를 그대로 인덱싱하므로 K-pop 신곡도 발매 당일부터 잡힌다.

In [5]:
import requests

ITUNES_SEARCH_URL = "https://itunes.apple.com/search"

def itunes_lookup(title: str, artist: str) -> Optional[dict]:
    params = {
        "term": f"{artist} {title}",
        "media": "music",
        "entity": "song",
        "limit": 5,
    }
    r = requests.get(ITUNES_SEARCH_URL, params=params, timeout=10)
    r.raise_for_status()
    results = r.json().get("results", [])
    if not results:
        return None
    title_lc = title.strip().lower()
    for hit in results:
        if hit.get("trackName", "").strip().lower() == title_lc:
            return hit
    return results[0]

def download_itunes_preview(title: str, artist: str) -> Optional[Path]:
    hit = itunes_lookup(title, artist)
    if hit is None or not hit.get("previewUrl"):
        return None
    safe = re.sub(r"[^\w\-]+", "_", f"{artist}_{title}").strip("_")
    out = TMP_AUDIO_DIR / f"{safe}.m4a"
    if not out.exists():
        with requests.get(hit["previewUrl"], timeout=30, stream=True) as resp:
            resp.raise_for_status()
            with open(out, "wb") as f:
                for chunk in resp.iter_content(8192):
                    f.write(chunk)
    return out

### 4-b. YouTube URL → yt-dlp (수동 입력 fallback)
iTunes에 없는 곡은 사용자가 YouTube URL을 직접 넘긴다.
자동 검색은 동명이곡을 잘못 잡을 수 있어 의도적으로 수동 입력만 받음.

In [6]:
def download_youtube_audio(url: str, basename: str) -> Path:
    import yt_dlp

    safe = re.sub(r"[^\w\-]+", "_", basename).strip("_")
    out_template = str(TMP_AUDIO_DIR / f"{safe}.%(ext)s")
    opts = {
        "format": "bestaudio[ext=m4a]/bestaudio",
        "outtmpl": out_template,
        "quiet": True,
        "no_warnings": True,
        "noprogress": True,
        "overwrites": False,
    }
    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(url, download=True)
        return Path(ydl.prepare_filename(info))

def acquire_audio(title: str, artist: str, youtube_url: Optional[str] = None) -> Path:
    """iTunes 1순위 → YouTube fallback. 둘 다 실패 시 RuntimeError."""
    p = download_itunes_preview(title, artist)
    if p is not None:
        print(f"[audio] iTunes preview: {p.name}")
        return p
    if youtube_url:
        p = download_youtube_audio(youtube_url, f"{artist}_{title}")
        print(f"[audio] YouTube: {p.name}")
        return p
    raise RuntimeError(
        f"음원 확보 실패: {title} - {artist}. iTunes에 없으면 youtube_url을 넘겨주세요."
    )

## 5. Librosa — 정량 feature

두 분기 공통으로 호출된다. Spotify 스타일의 0~1 스케일로 대충 정규화해서
프롬프트가 읽기 쉽도록 만든다.

In [7]:
def extract_audio_features(audio_path: str | Path) -> dict:
    import numpy as np
    import librosa

    y, sr = librosa.load(str(audio_path), mono=True)
    duration = float(librosa.get_duration(y=y, sr=sr))
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    tempo = float(np.atleast_1d(tempo)[0])  # librosa>=0.10 returns ndarray
    rms = float(np.mean(librosa.feature.rms(y=y)))
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y=y)))
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    dynamics_range = float(np.percentile(onset_env, 95) - np.percentile(onset_env, 5))
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    key_idx = int(chroma.mean(axis=1).argmax())
    harmonic, percussive = librosa.effects.hpss(y)
    h_energy = float(np.mean(np.abs(harmonic)))
    p_energy = float(np.mean(np.abs(percussive)))
    harmonic_ratio = h_energy / (h_energy + p_energy + 1e-9)

    energy = float(min(1.0, rms * 8))
    valence = float(min(1.0, max(0.0, (centroid - 1000) / 4000)))
    danceability = float(min(1.0, max(0.0, 1.0 - zcr * 3)))

    return {
        "duration": duration,
        "tempo": tempo,
        "energy": energy,
        "valence": valence,
        "danceability": danceability,
        "key": key_idx,
        "harmonic_ratio": float(harmonic_ratio),
        "dynamics_range": dynamics_range,
        "_source": "librosa",
    }

## 6. Genius — 분기 판정 + narrative text

In [8]:
from lyricsgenius import Genius

_genius_client: Optional[Genius] = None

def get_genius_client() -> Genius:
    global _genius_client
    if _genius_client is None:
        if not GENIUS_ACCESS_TOKEN:
            raise RuntimeError("GENIUS_ACCESS_TOKEN가 .env에 없음")
        client = Genius(
            GENIUS_ACCESS_TOKEN,
            timeout=10,
            retries=2,
            remove_section_headers=True,
            skip_non_songs=True,
        )
        client.verbose = False
        _genius_client = client
    return _genius_client

def fetch_genius_song(title: str, artist: str):
    """Genius song 객체. 못 찾으면 None."""
    g = get_genius_client()
    try:
        return g.search_song(title, artist)
    except Exception as e:
        print(f"[warn] Genius 조회 실패: {e}")
        return None

def build_narrative_text(song, lyrics_chars: int = 1500) -> str:
    if song is None:
        return ""
    lyrics = re.sub(r"\n{3,}", "\n\n", (song.lyrics or "").strip())[:lyrics_chars]
    return "\n".join([
        f"Title: {song.title}",
        f"Artist: {song.artist}",
        f"Album: {getattr(song, 'album', '') or ''}",
        f"Lyrics excerpt:\n{lyrics}",
    ])

## 7. Gemini 호출 — text / audio 두 모드

In [9]:
from google import genai
from google.genai import types

_gemini_client = None
GEMINI_TEXT_MODEL = "gemini-2.5-flash"
GEMINI_AUDIO_MODEL = "gemini-2.5-pro"

def get_gemini_client():
    global _gemini_client
    if _gemini_client is None:
        if not GEMINI_API_KEY:
            raise RuntimeError("GEMINI_API_KEY가 .env에 없음")
        _gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    return _gemini_client

def _parse_json_response(raw: str) -> dict:
    raw = raw.strip()
    m = re.search(r"\{.*\}", raw, re.DOTALL)
    if m is None:
        raise ValueError(f"Gemini 응답에서 JSON을 찾지 못함:\n{raw}")
    return json.loads(m.group(0))

def gemini_classify_text(features: dict, narrative_text: str, title: str, artist: str) -> dict:
    client = get_gemini_client()
    user_msg = (
        f"Song: {title} - {artist}\n"
        f"Librosa features: {json.dumps(features, ensure_ascii=False)}\n"
        f"Narrative text:\n{narrative_text}"
    )
    resp = client.models.generate_content(
        model=GEMINI_TEXT_MODEL,
        contents=user_msg,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            response_mime_type="application/json",
            temperature=0.3,
        ),
    )
    return _parse_json_response(resp.text)

def gemini_classify_audio(features: dict, audio_path: Path, title: str, artist: str) -> dict:
    client = get_gemini_client()
    audio_bytes = audio_path.read_bytes()
    mime = "audio/mp4" if audio_path.suffix.lower() in {".m4a", ".mp4"} else "audio/mpeg"
    user_text = (
        f"Song: {title} - {artist}\n"
        f"Librosa features: {json.dumps(features, ensure_ascii=False)}\n"
        f"가사/메타가 없는 신곡이다. 첨부된 오디오를 직접 들어 분류하라."
    )
    resp = client.models.generate_content(
        model=GEMINI_AUDIO_MODEL,
        contents=[
            types.Part.from_bytes(data=audio_bytes, mime_type=mime),
            user_text,
        ],
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            response_mime_type="application/json",
            temperature=0.3,
        ),
    )
    return _parse_json_response(resp.text)

## 8. 두 분기 + 라우터

- `process_known_song()` — Genius hit. librosa + Genius narrative → text 분류
- `process_new_song()` — Genius miss. librosa + 오디오 → audio 분류
- `process_kpop_visualization()` — 캐시 → Genius 시도 → 분기 라우팅

In [10]:
def process_known_song(
    title: str, artist: str, song, *, youtube_url: Optional[str] = None,
) -> dict:
    audio_path = acquire_audio(title, artist, youtube_url=youtube_url)
    features = extract_audio_features(audio_path)
    narrative = build_narrative_text(song)
    visual_params = gemini_classify_text(features, narrative, title, artist)
    return {
        "metadata": {
            "title": title,
            "artist": artist,
            "branch": "known",
            "genius_url": getattr(song, "url", None),
            "audio_source": str(audio_path.name),
        },
        "live_features": features,
        "concept_features": visual_params,
    }

def process_new_song(
    title: str, artist: str, *, youtube_url: Optional[str] = None,
) -> dict:
    audio_path = acquire_audio(title, artist, youtube_url=youtube_url)
    features = extract_audio_features(audio_path)
    visual_params = gemini_classify_audio(features, audio_path, title, artist)
    return {
        "metadata": {
            "title": title,
            "artist": artist,
            "branch": "new",
            "audio_source": str(audio_path.name),
        },
        "live_features": features,
        "concept_features": visual_params,
    }

def process_kpop_visualization(
    title: str,
    artist: str,
    *,
    youtube_url: Optional[str] = None,
    force_refresh: bool = False,
) -> dict:
    if not force_refresh:
        cached = get_from_cache(title, artist)
        if cached is not None:
            print(f"[cache hit] {title} - {artist}")
            return cached

    song = fetch_genius_song(title, artist)
    if song is not None:
        print(f"[branch] known — Genius hit: {song.url}")
        result = process_known_song(title, artist, song, youtube_url=youtube_url)
    else:
        print(f"[branch] new — Genius miss")
        result = process_new_song(title, artist, youtube_url=youtube_url)

    save_to_cache(result)
    return result

## 9. 실행 예시

In [11]:
# 아는 곡 — Genius hit 예상
result = process_kpop_visualization("Supernova", "aespa")
print(json.dumps(result, ensure_ascii=False, indent=2))

[branch] known — Genius hit: https://genius.com/Aespa-supernova-lyrics
[audio] iTunes preview: aespa_Supernova.m4a


/Users/yuli/Documents/GitHub/0_Exp/Spatial_Audio_Visualization/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/xt/9007rybs7kdb25r3g6xbrq840000gn/T/ipykernel_47434/462960830.py:5: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(str(audio_path), mono=True)
/Users/yuli/Documents/GitHub/0_Exp/Spatial_Audio_Visualization/.venv/lib/python3.11/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


{
  "metadata": {
    "title": "Supernova",
    "artist": "aespa",
    "branch": "known",
    "genius_url": "https://genius.com/Aespa-supernova-lyrics",
    "audio_source": "aespa_Supernova.m4a"
  },
  "live_features": {
    "duration": 30.000453514739228,
    "tempo": 117.45383522727273,
    "energy": 1.0,
    "valence": 0.33766925342958043,
    "danceability": 0.7609336481293504,
    "key": 4,
    "harmonic_ratio": 0.6665787818700852,
    "dynamics_range": 4.091363430023193,
    "_source": "librosa"
  },
  "concept_features": {
    "sonic_texture": "Synth-pop",
    "narrative_archetype": "Cyberpunk",
    "visual_symbols": [
      "Supernova",
      "Tick-tick bomb",
      "Portal"
    ],
    "color_palette": {
      "main": "#8A2BE2",
      "sub": [
        "#FFD700",
        "#000033"
      ]
    },
    "reasoning_brief": "높은 에너지와 신스 사운드를 기반으로 한 'Synth-pop' 텍스처를 지닌다. 가사의 우주적, 미래적, 폭발적 요소는 'Cyberpunk' 서사를 형성하며, 슈퍼노바, 시한폭탄, 포털이 주요 시각 상징이다. 색상은 높은 에너지와 중간 정도의 밸런스를 반영하여 깊은 일렉트릭 퍼플을 메인으로

In [15]:
# 신곡 + iTunes에 없는 곡 — YouTube URL 수동 지정
result = process_kpop_visualization(
    "사랑쪽지",
    "세븐틴",
    youtube_url="https://youtu.be/fJquWD13T0I?si=tlYafBt-S22CMyIy",
 )

print(json.dumps(result, ensure_ascii=False, indent=2))

[branch] known — Genius hit: https://genius.com/Seventeen-love-letter-lyrics
[audio] iTunes preview: 세븐틴_사랑쪽지.m4a


/var/folders/xt/9007rybs7kdb25r3g6xbrq840000gn/T/ipykernel_47434/462960830.py:5: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(str(audio_path), mono=True)
/Users/yuli/Documents/GitHub/0_Exp/Spatial_Audio_Visualization/.venv/lib/python3.11/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


{
  "metadata": {
    "title": "사랑쪽지",
    "artist": "세븐틴",
    "branch": "known",
    "genius_url": "https://genius.com/Seventeen-love-letter-lyrics",
    "audio_source": "세븐틴_사랑쪽지.m4a"
  },
  "live_features": {
    "duration": 30.000453514739228,
    "tempo": 89.10290948275862,
    "energy": 1.0,
    "valence": 0.40709018115940226,
    "danceability": 0.6514136365284222,
    "key": 4,
    "harmonic_ratio": 0.582876290072504,
    "dynamics_range": 6.144679546356201,
    "_source": "librosa"
  },
  "concept_features": {
    "sonic_texture": "Synth-pop",
    "narrative_archetype": "High-teen",
    "visual_symbols": [
      "쪽지",
      "하늘",
      "펜"
    ],
    "color_palette": {
      "main": "#7FCFFC",
      "sub": [
        "#FFDAB9",
        "#F5F5DC"
      ]
    },
    "reasoning_brief": "제목과 가사에서 드러나는 풋풋하고 진심 어린 고백, 미래에 대한 희망이 'High-teen' 내러티브와 잘 맞습니다. 높은 에너지와 하늘, 바람 등의 시각적 요소는 밝고 경쾌한 'Synth-pop' 질감과 연결됩니다. 쪽지, 하늘, 펜은 가사의 핵심적인 시각적 상징이며, 색상은 높은 에너지와 희망적인 분위기를 반영하되, 약간의 서툰 감성을 담아 밝고